# Task 1: Data Preprocessing and Exploratory Data Analysis

## Objective

The objective of this task is to collect, preprocess, and analyze historical financial data for Tesla (TSLA), Vanguard Total Bond Market ETF (BND), and SPDR S&P 500 ETF (SPY). This exploratory analysis provides the foundation for building reliable forecasting models in later stages of the project.

## Dataset

Historical market data is obtained from the Yahoo Finance API using the `yfinance` Python package.

Assets analyzed:

- **TSLA** – Tesla Inc.
- **BND** – Vanguard Total Bond Market ETF
- **SPY** – SPDR S&P 500 ETF

Analysis period:

**January 1, 2015 – June 30, 2026**

This notebook covers:

- Data extraction
- Data cleaning
- Exploratory Data Analysis (EDA)
- Stationarity testing
- Volatility analysis
- Risk metrics (Sharpe Ratio and Value at Risk)

## Import Required Libraries

The following libraries are used for data manipulation, visualization, statistical analysis, and downloading financial data.

In [1]:
# Data manipulation
import numpy as np
import pandas as pd

# Data acquisition
import yfinance as yf

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy.stats import zscore
from statsmodels.tsa.stattools import adfuller

# Notebook settings
plt.style.use("seaborn-v0_8")
sns.set_theme(style="whitegrid")

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

## Download Historical Market Data

Historical daily market data is collected using the Yahoo Finance API through the `yfinance` package.

Assets:

- Tesla (TSLA)
- Vanguard Total Bond Market ETF (BND)
- SPDR S&P 500 ETF (SPY)

The data spans from **January 1, 2015** to **June 30, 2026**.

In [2]:
START_DATE = "2015-01-01"
END_DATE = "2026-06-30"

tickers = {
    "TSLA": "Tesla",
    "BND": "Vanguard Total Bond Market ETF",
    "SPY": "SPDR S&P 500 ETF"
}

data = {}

for ticker in tickers:
    df = yf.download(
        ticker,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=False,
        progress=False
    )

    df["Ticker"] = ticker
    data[ticker] = df

print("Data downloaded successfully.")

Data downloaded successfully.


In [3]:
for ticker, df in data.items():
    print(f"\n{'='*50}")
    print(f"{ticker}")
    print(df.head())


TSLA
Price      Adj Close   Close    High     Low    Open    Volume Ticker
Ticker          TSLA    TSLA    TSLA    TSLA    TSLA      TSLA       
Date                                                                 
2015-01-02   14.6207 14.6207 14.8833 14.2173 14.8580  71466000   TSLA
2015-01-05   14.0060 14.0060 14.4333 13.8107 14.3033  80527500   TSLA
2015-01-06   14.0853 14.0853 14.2800 13.6140 14.0040  93928500   TSLA
2015-01-07   14.0633 14.0633 14.3187 13.9853 14.2233  44526000   TSLA
2015-01-08   14.0413 14.0413 14.2533 14.0007 14.1873  51637500   TSLA

BND
Price      Adj Close   Close    High     Low    Open   Volume Ticker
Ticker           BND     BND     BND     BND     BND      BND       
Date                                                                
2015-01-02   59.2057 82.6500 82.6900 82.4200 82.4300  2218800    BND
2015-01-05   59.3775 82.8900 82.9200 82.7000 82.7400  5820100    BND
2015-01-06   59.5495 83.1300 83.3800 83.0300 83.0300  3887600    BND
2015-01-07   59

In [4]:
import os

os.makedirs("../data/raw", exist_ok=True)

for ticker, df in data.items():
    df.to_csv(f"../data/raw/{ticker}.csv")

print("Raw datasets saved successfully.")

Raw datasets saved successfully.


## Data Inspection and Cleaning

Before performing exploratory analysis, the datasets are inspected to ensure data quality. This includes:

- Examining dataset dimensions
- Checking data types
- Identifying missing values
- Detecting duplicate records
- Flattening MultiIndex columns returned by `yfinance`
- Verifying chronological ordering

In [5]:
# Flatten MultiIndex columns if present
for ticker, df in data.items():
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    data[ticker] = df

In [6]:
for ticker, df in data.items():
    print(f"\n{ticker}")
    print(df.columns.tolist())


TSLA
['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']

BND
['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']

SPY
['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']


In [7]:
for ticker, df in data.items():
    print("=" * 60)
    print(ticker)
    print(f"Shape: {df.shape}")
    display(df.head())

TSLA
Shape: (2888, 7)


Price,Adj Close,Close,High,Low,Open,Volume,Ticker
Date,,,,,,,
2015-01-02,14.6207,14.6207,14.8833,14.2173,14.8580,71466000,TSLA
2015-01-05,14.0060,14.0060,14.4333,13.8107,14.3033,80527500,TSLA
2015-01-06,14.0853,14.0853,14.2800,13.6140,14.0040,93928500,TSLA
2015-01-07,14.0633,14.0633,14.3187,13.9853,14.2233,44526000,TSLA
2015-01-08,14.0413,14.0413,14.2533,14.0007,14.1873,51637500,TSLA


BND
Shape: (2888, 7)


Price,Adj Close,Close,High,Low,Open,Volume,Ticker
Date,,,,,,,
2015-01-02,59.2057,82.6500,82.6900,82.4200,82.4300,2218800,BND
2015-01-05,59.3775,82.8900,82.9200,82.7000,82.7400,5820100,BND
2015-01-06,59.5495,83.1300,83.3800,83.0300,83.0300,3887600,BND
2015-01-07,59.5853,83.1800,83.2800,83.0500,83.1400,2433400,BND
2015-01-08,59.4922,83.0500,83.1100,82.9700,83.1100,1873400,BND


SPY
Shape: (2888, 7)


Price,Adj Close,Close,High,Low,Open,Volume,Ticker
Date,,,,,,,
2015-01-02,169.6879,205.4300,206.8800,204.1800,206.3800,121465900,SPY
2015-01-05,166.6233,201.7200,204.3700,201.3500,204.1700,169632600,SPY
2015-01-06,165.0539,199.8200,202.7200,198.8600,202.0900,209151400,SPY
2015-01-07,167.1107,202.3100,202.7200,200.8800,201.4200,125346700,SPY
2015-01-08,170.0761,205.9000,206.1600,203.9900,204.0100,147217800,SPY


In [8]:
for ticker, df in data.items():
    print("=" * 60)
    print(ticker)
    print(df.dtypes)

TSLA
Price
Adj Close    float64
Close        float64
High         float64
Low          float64
Open         float64
Volume         int64
Ticker           str
dtype: object
BND
Price
Adj Close    float64
Close        float64
High         float64
Low          float64
Open         float64
Volume         int64
Ticker           str
dtype: object
SPY
Price
Adj Close    float64
Close        float64
High         float64
Low          float64
Open         float64
Volume         int64
Ticker           str
dtype: object


In [9]:
for ticker, df in data.items():
    print("=" * 60)
    print(ticker)
    print(df.isnull().sum())

TSLA
Price
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
Ticker       0
dtype: int64
BND
Price
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
Ticker       0
dtype: int64
SPY
Price
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
Ticker       0
dtype: int64


In [10]:
for ticker, df in data.items():
    duplicates = df.duplicated().sum()
    print(f"{ticker}: {duplicates} duplicate rows")

TSLA: 0 duplicate rows
BND: 0 duplicate rows
SPY: 0 duplicate rows


In [11]:
for ticker, df in data.items():
    print("=" * 60)
    print(ticker)
    display(df.describe())

TSLA


Price,Adj Close,Close,High,Low,Open,Volume
count,2888.0000,2888.0000,2888.0000,2888.0000,2888.0000,2888.0000
mean,148.7739,148.7739,151.9906,145.4167,148.7973,108792194.4598
std,138.8960,138.8960,141.8528,135.8672,138.9772,70825493.1529
min,9.5780,9.5780,10.3313,9.4033,9.4880,10620000.0000
25%,18.3935,18.3935,18.6652,18.0237,18.3908,65483250.0000
50%,133.4377,133.4377,136.0533,125.8317,131.4963,90336150.0000
75%,251.9258,251.9258,257.4850,245.8325,251.6800,126120450.0000
max,489.8800,489.8800,498.8300,485.3300,489.8800,914082000.0000


BND


Price,Adj Close,Close,High,Low,Open,Volume
count,2888.0000,2888.0000,2888.0000,2888.0000,2888.0000,2888.0000
mean,66.2811,79.3277,79.4421,79.2112,79.3307,4653785.8033
std,4.7001,5.3104,5.2980,5.3225,5.3138,3017703.9497
min,58.5318,68.0400,68.3800,67.9900,68.0800,0.0000
25%,62.2680,73.8000,73.9200,73.6400,73.8000,2233700.0000
50%,65.5074,80.8150,80.9050,80.7100,80.8000,4280650.0000
75%,70.4514,83.4400,83.5500,83.3325,83.4700,6246475.0000
max,74.5813,89.4800,89.5900,89.4400,89.5500,33963000.0000


SPY


Price,Adj Close,Close,High,Low,Open,Volume
count,2888.0000,2888.0000,2888.0000,2888.0000,2888.0000,2888.0000
mean,351.5055,375.2167,377.1672,372.9466,375.1495,85510494.9792
std,155.4439,146.5847,147.2447,145.7748,146.5589,43385532.6469
min,154.1616,182.8600,184.1000,181.0200,182.3400,20270000.0000
25%,223.5468,254.5600,255.9200,252.4775,254.5775,58364550.0000
50%,312.8179,339.4350,342.3150,337.1650,339.8350,75419500.0000
75%,432.8068,453.6775,456.0000,451.5500,453.9900,98822450.0000
max,757.6182,759.5700,760.4000,756.7500,758.1500,507244300.0000


In [12]:
for ticker, df in data.items():
    print(
        f"{ticker}: "
        f"{df.index.min().date()} → {df.index.max().date()}"
    )

TSLA: 2015-01-02 → 2026-06-29
BND: 2015-01-02 → 2026-06-29
SPY: 2015-01-02 → 2026-06-29
